## Import libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# advanced imputers
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


## Load and observe data

In [ ]:
df = pd.read_csv('insurance.csv')

In [ ]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
df.dtypes

,0
age,int64
sex,object
bmi,float64
children,int64
smoker,object
region,object
charges,float64


## Handling categorical columns

In [ ]:
df['smoker'] = df['smoker'].map({'yes':1,'no':0})
df['sex'] = df['sex'].map({'male':1,'female':0})
df = pd.get_dummies(df,columns=['region'],dtype=int,drop_first=True)
df.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0,0,1
1,18,1,33.770,1,0,1725.55230,0,1,0
2,28,1,33.000,3,0,4449.46200,0,1,0
3,33,1,22.705,0,0,21984.47061,1,0,0
4,32,1,28.880,0,0,3866.85520,1,0,0


## Introducing Missing values for experimentation

In [ ]:
df.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
charges,0
region_northwest,0
region_southeast,0
region_southwest,0


In [ ]:
np.random.seed(42)
df.loc[df.sample(frac = 0.3).index,'bmi'] = np.nan

## Handling and comparing missing value imputation with different techniques

In [ ]:
mean_imputer = SimpleImputer(strategy='mean')
median_imputer = SimpleImputer(strategy='median')
knn_imputer = KNNImputer(n_neighbors=5)
iterative_imputer = IterativeImputer(estimator= LinearRegression(),random_state=42)

In [ ]:
#Mean imputation
df_mean  = df.copy()
df_mean['bmi'] = mean_imputer.fit_transform(df_mean[['bmi']])

#Median imputation
df_median  = df.copy()
df_median['bmi'] = median_imputer.fit_transform(df_median[['bmi']])

#KNN imputation
df_knn  = df.copy()
df_knn = pd.DataFrame(knn_imputer.fit_transform(df_knn),columns = df.columns)

#Iterative imputation
df_iterative  = df.copy()
df_iterative = pd.DataFrame(iterative_imputer.fit_transform(df_iterative),columns = df.columns)

In [ ]:
#Train and evaluate model
def evaluate_impute_strategy(df_input,name):
  x = df_input.drop('charges',axis = 1)
  y = df_input['charges']
  x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state= 42)
  model = LinearRegression()
  model.fit(x_train,y_train)
  return {"Strategy":name,"R2_score":r2_score(y_test,model.predict(x_test))}

In [ ]:
evaluate_impute_strategy(df_mean,"Mean Imputation")

{'Strategy': 'Mean Imputation', 'R2_score': 0.7592025798815374}

In [ ]:
evaluate_impute_strategy(df_median,"Median Imputation")

{'Strategy': 'Median Imputation', 'R2_score': 0.7591939972478123}

In [ ]:
evaluate_impute_strategy(df_knn,"KNN Imputation")

{'Strategy': 'KNN Imputation', 'R2_score': 0.7775569903381638}

In [ ]:
evaluate_impute_strategy(df_iterative,"Iterative Imputation")

{'Strategy': 'Iterative Imputation', 'R2_score': 0.805805232643733}

### My Observations
  * **Mean/Mode Imputation**:
      * Fast calcuation as we are just calculating the mean and median of the sepecific column.
      * Low r2-score becuase it doesn't consider the relation of bmi with other data (age,sex etc) its just the average/median value , making it difficult for the model to understand the exact relation with the target variable "Charges".

  * **KNNImputer/IterativeImputer**:
      * Computationally expensive than Mean/Mode imputation,as we have to run mahcine learning algorithm to predict the missing values.
      * Better result because it uses the 'Context' of a person's other data(age,sex etc) to predict the missing values of 'bmi'.

## Handling Outliers and comparing different techniqes

In [ ]:
#Without handling outliers
df_base = df_mean.copy()

#Percentile method
df_percentile = df_mean.copy()
upper_limit_percentile = df_percentile['charges'].quantile(0.95)
df_percentile['charges'] = np.where(df_percentile['charges'] > upper_limit_percentile,upper_limit_percentile,df_percentile['charges'])

#IQR method
df_iqr = df_iterative.copy()
Q1 = df_iqr['charges'].quantile(0.25)
Q3 = df_iqr['charges'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5*IQR
upper_limit = Q3 + 1.5*IQR

df_iqr['charges'] = np.where(df_iqr['charges'] > upper_limit,upper_limit,np.where(df_iqr['charges'] < lower_limit,lower_limit,df_iqr['charges']))


In [ ]:
#Train and evaluate model
def evaluate_outlier_strategy(df_input,name):
  x = df_input.drop('charges',axis = 1)
  y = df_input['charges']
  x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state = 42)

  model = LinearRegression()
  model.fit(x_train,y_train)
  return {'Strategy':name,'R2_score':r2_score(y_test,model.predict(x_test))}

In [ ]:
evaluate_outlier_strategy(df_base,"Without Handling Outliers")

{'Strategy': 'Without Handling Outliers', 'R2_score': 0.7592025798815374}

In [ ]:
evaluate_outlier_strategy(df_percentile,"Percentile Method")

{'Strategy': 'Percentile Method', 'R2_score': 0.7718988400368081}

In [ ]:
evaluate_outlier_strategy(df_iqr,"IQR Method")

{'Strategy': 'IQR Method', 'R2_score': 0.8089216093390206}

### My Observations
* **Without Removing Outliers**:
  * The error (MSE) for outliers will be very large as compared to others,so these outliers will try pull the regression line towards them ( in order to minimize the error) making it less accurate for majority of the persons hence results in bad performance.

* **Percentile Method**:
  * Capping the value of charges at 95% means treating the top 5% of data as outliers ( it is a bit random).
  * CASE1: If 10% of our data are actually outliers we are only handling the 5% of the outliers.
  * CASE2: If 1% of our data are actually outliers we are losing the 4% percent of our perfectly good data.

* **Inter Quartile Range Method**:
  * In this method instead of picking a random percentage, we use the spread of the middle 50% (IQR = Q3-Q1) of the data to define what normal looks like and what should be outliers.
  * It adapts according to specific data and clip values that are statiscally extreme.